# Demo for Section 2 - Sorting, Searching and Ranking

This notebook demonstrates all public functions in `numcompute/sort_search.py`
and `numcompute/rank.py` using small, reproducible examples.

Open it in Jupyter or VS Code, select a Python kernel with NumPy installed, and
choose **Run All**. The working directory can be the project root or `demo/`.
If needed, install NumPy in your selected kernel with `%pip install numpy` in a
separate cell. No datasets or plotting packages are required.

We will sort values and recover their original indices, select extreme values,
find insertion positions, compare tie methods, and calculate percentiles.
Assertions check the examples as they run. Saved outputs show a previous run;
running all cells refreshes them for your environment.

In [1]:
from pathlib import Path
import platform
import sys
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "numcompute").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "numcompute").is_dir():
    raise RuntimeError("Run this notebook from the project root or its demo folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from numcompute.sort_search import (
    merge_sort, sort, argsort, quickselect, topk, binary_search, searchsorted,
)
from numcompute.rank import rank, percentile, percentile_rank

np.set_printoptions(precision=3, suppress=True)
print("Python:", platform.python_version())
print("NumPy:", np.__version__)

Python: 3.12.13
NumPy: 1.26.4


## 1. Stable sorting and original indices

`merge_sort` is a from-scratch, stable merge sort for 1-D arrays. Equal values
keep their input order. With `return_indices=True`, it also returns the original
positions, so we can reorder related labels. `sort` and `argsort` are NumPy-backed
alternatives; both are stable by default. These functions leave the input unchanged.

In [ ]:
names = np.array(["Ava", "Ben", "Chen", "Dia", "Eli", "Faye"])
scores = np.array([88, 72, 88, 95, 60, 72])
original_scores = scores.copy()

ordered, order = merge_sort(scores, return_indices=True)
print("Original scores:", scores)
print("Sorted scores:  ", ordered)
print("Original indices:", order)
print("Names in order: ", names[order])

np.testing.assert_array_equal(order, [4, 1, 5, 0, 2, 3])
np.testing.assert_array_equal(scores[order], ordered)
np.testing.assert_array_equal(sort(scores), ordered)
np.testing.assert_array_equal(argsort(scores), order)
np.testing.assert_array_equal(scores, original_scores)

## 2. Sorting along an axis

For a matrix, `axis=1` sorts across each row and `axis=0` sorts down each column.
The default `axis=-1` means the last axis. Indices returned by `argsort` refer to
positions along the chosen axis; use `np.take_along_axis` to recover the values.

In [ ]:
matrix = np.array([[30, 10, 20], [60, 40, 50], [15, 35, 25]])
row_order = argsort(matrix, axis=1)
print("Input:\n", matrix)
print("Each row sorted:\n", sort(matrix, axis=1))
print("Each column sorted:\n", sort(matrix, axis=0))
print("Sorting indices for each row:\n", row_order)

np.testing.assert_array_equal(
    np.take_along_axis(matrix, row_order, axis=1), sort(matrix, axis=1)
)

## 3. Quickselect: one order statistic

`quickselect(x, k)` returns the value at **zero-based position** `k` in sorted
data, without fully sorting it. Here `k=2` means the third smallest value, 72.
The seed makes pivot choices reproducible; the selected value does not depend
on that seed. Quickselect requires 1-D data without NaNs.

In [ ]:
k = 2
selected = quickselect(scores, k, rng=17)
print(f"Value at sorted position {k}: {selected}")
print("Smallest:", quickselect(scores, 0, rng=17))
print("Largest: ", quickselect(scores, len(scores) - 1, rng=17))
assert selected == ordered[k] == 72
np.testing.assert_array_equal(scores, original_scores)

## 4. Top-k values and their indices

For `topk`, **k is a count**, rather than a position. By default it returns the
largest values in descending order. `largest=False` selects the smallest values.
Ties favour the smaller original index, including at the selection cutoff:
when selecting two scores, Ava's 88 is chosen before Chen's 88.

With `sorted=False`, the selected values need not be ordered. Use this when only
membership matters. Matrix indices again refer to positions along the chosen axis.

In [ ]:
best, best_indices = topk(scores, 2)
lowest, lowest_indices = topk(scores, 3, largest=False)
unordered, unordered_indices = topk(scores, 3, sorted=False)
row_best, row_best_indices = topk(matrix, 2, axis=1)

print("Top two:", list(zip(names[best_indices].tolist(), best.tolist())))
print("Lowest three:", lowest, "at indices", lowest_indices)
print("Top three, order not requested:", unordered)
print("Top two per row:\n", row_best)
print("Their column indices:\n", row_best_indices)

np.testing.assert_array_equal(best_indices, [3, 0])
np.testing.assert_array_equal(lowest, [60, 72, 72])
np.testing.assert_array_equal(np.sort(unordered), [88, 88, 95])
np.testing.assert_array_equal(scores[unordered_indices], unordered)
np.testing.assert_array_equal(
    np.take_along_axis(matrix, row_best_indices, axis=1), row_best
)

## 5. Searching returns insertion positions

Both search functions require **ascending input**; they do not sort or check it.
`side="left"` inserts before equal values, and `side="right"` inserts after them.
Their difference counts occurrences of a target. A position can equal the array's
length, so check the bounds before indexing when testing for an exact match.

In [ ]:
target = 72
left = binary_search(ordered, target, side="left")
right = binary_search(ordered, target, side="right")
print("Sorted scores:", ordered)
print(f"{target}: left={left}, right={right}, count={right - left}")

queries = np.array([50, 72, 80, 100])
positions = searchsorted(ordered, queries)
print("Queries:        ", queries)
print("Left positions: ", positions)
print("Right positions:", searchsorted(ordered, queries, side="right"))
for value, position in zip(queries, positions):
    found = position < ordered.size and ordered[position] == value
    print(f"{value}: insertion position {position}, exact match: {bool(found)}")

assert right - left == 2
np.testing.assert_array_equal(positions, [0, 1, 3, 6])
np.testing.assert_array_equal(
    positions, [binary_search(ordered, value) for value in queries]
)

## 6. Ranking with ties

Ranks are **1-based**, with the smallest value ranked first. Results stay in
the original input order and use floats to represent averages and NaNs.

| Method | How tied positions are assigned |
|---|---|
| `average` | Average of the occupied ranks |
| `min` | Lowest occupied rank |
| `max` | Highest occupied rank |
| `dense` | One rank per distinct value, without gaps |
| `ordinal` | Separate ranks, preserving input order within ties |

In [ ]:
tied = np.array([30, 10, 20, 20])
expected_ranks = {
    "average": [4, 1, 2.5, 2.5],
    "min": [4, 1, 2, 2],
    "max": [4, 1, 3, 3],
    "dense": [3, 1, 2, 2],
    "ordinal": [4, 1, 2, 3],
}
print("Input:", tied)
for method, expected in expected_ranks.items():
    result = rank(tied, method=method)
    print(f"{method:8s}: {result}")
    np.testing.assert_allclose(result, expected)

## 7. Ranking matrices and missing values

`rank` defaults to the last axis. Both `nan_policy="keep"` and `"omit"` exclude
NaNs from the ranking and leave their positions as NaN. Valid observations are
ranked among themselves. An all-NaN slice remains all NaN. Use `"raise"` to reject
missing values (demonstrated in the validation section).

In [ ]:
missing = np.array([[30., np.nan, 10., 10.], [np.nan, np.nan, np.nan, np.nan]])
kept = rank(missing, axis=1, nan_policy="keep")
omitted = rank(missing, axis=1, nan_policy="omit")
print("Input with missing values:\n", missing)
print("Row ranks:\n", kept)
print("Column ranks for the earlier matrix:\n", rank(matrix, axis=0))

np.testing.assert_allclose(kept, omitted, equal_nan=True)
np.testing.assert_allclose(kept[0], [3, np.nan, 1.5, 1.5], equal_nan=True)
assert np.isnan(kept[1]).all()

## 8. Percentiles and interpolation

`percentile(x, q)` takes percentages from 0 to 100. Its default `axis=None`
flattens the data. A scalar `q` gives one percentile; multiple requests add their
shape before the remaining data dimensions.

For `[10, 20, 30, 40]`, the 25th percentile lies at sorted position
`(25 / 100) * (4 - 1) = 0.75`. Linear interpolation gives 17.5. Other methods
choose the lower, higher or nearest observation, or average the two endpoints.

In [ ]:
sample = np.array([10, 20, 30, 40])
expected_percentiles = {
    "linear": 17.5, "lower": 10, "higher": 20, "nearest": 20, "midpoint": 15,
}
for method, expected in expected_percentiles.items():
    result = percentile(sample, 25, method=method)
    print(f"25th percentile ({method:8s}): {result}")
    assert result == expected

q = [0, 25, 50, 75, 100]
summary = percentile(scores, q)
per_column = percentile(matrix, [25, 50, 75], axis=0)
print("Requested percentages:", q)
print("Score percentiles:    ", summary)
print("25th, 50th, 75th percentile per column:\n", per_column)
print("Result shape (percentiles, columns):", per_column.shape)
np.testing.assert_allclose(summary, np.percentile(scores, q))
np.testing.assert_allclose(per_column, np.percentile(matrix, [25, 50, 75], axis=0))

## 9. Percentiles with NaNs

The default `nan_policy="propagate"` returns NaN for a slice containing a NaN.
`"omit"` computes percentiles from the available values. Even with omission, an
all-NaN slice returns NaN. Unlike `rank`, percentiles reduce the selected axis.

In [ ]:
readings = np.array([[10., np.nan, 30.], [20., 40., 60.], [np.nan, np.nan, np.nan]])
propagated = percentile(readings, 50, axis=1)
available = percentile(readings, 50, axis=1, nan_policy="omit")
print("Row medians, propagate:", propagated)
print("Row medians, omit:     ", available)
np.testing.assert_allclose(propagated, [np.nan, 40, np.nan], equal_nan=True)
np.testing.assert_allclose(available, [20, 40, np.nan], equal_nan=True)

## 10. Percentile rank: where does a score stand?

`percentile` maps a percentage to a data value; `percentile_rank` maps a score
to a percentage. With ties and interpolation, these are not exact inverses.
For `[10, 20, 20, 30]` and a score of 20:

- `weak`: 3 of 4 values are at or below 20, giving 75%.
- `strict`: 1 of 4 values is below 20, giving 25%.
- `mean`: the average of those percentages, giving 50%.

The input data must be nonempty and 1-D. A NaN in the data propagates to every
result; a NaN query affects only that query.

In [ ]:
reference = np.array([10, 20, 20, 30])
for kind, expected in [("weak", 75), ("strict", 25), ("mean", 50)]:
    result = percentile_rank(reference, 20, kind=kind)
    print(f"Score 20, {kind:6s}: {result}%")
    assert result == expected

query_scores = np.array([5., 20., 25., 35., np.nan])
standing = percentile_rank(reference, query_scores)
print("Query scores:     ", query_scores)
print("Weak standings (%):", standing)
print("NaN in reference:", percentile_rank([10, np.nan, 30], [20, 30]))
np.testing.assert_allclose(standing, [0, 75, 75, 100, np.nan], equal_nan=True)

## 11. Combining the modules: a score report

Use `topk` to choose the three highest scores, `rank` to assign positions, and
`percentile_rank` to show each student's standing. Negating these finite scores
makes the highest score receive rank 1. The `min` method gives tied students
the same competition rank, with a gap after the tie.

In [ ]:
competition_ranks = rank(-scores, method="min")
standings = percentile_rank(scores, scores, kind="weak")
winning_scores, winning_indices = topk(scores, 3)
print(f"{'Name':<8} {'Score':>6} {'Rank':>6} {'At or below (%)':>16}")
for index in winning_indices:
    print(f"{names[index]:<8} {scores[index]:>6} {competition_ranks[index]:>6.0f} {standings[index]:>16.1f}")
print("Median score:", percentile(scores, 50))
np.testing.assert_array_equal(winning_indices, [3, 0, 2])
np.testing.assert_allclose(competition_ranks[winning_indices], [1, 2, 2])
np.testing.assert_array_equal(scores, original_scores)

## 12. Boundaries and helpful validation errors

Sorting an empty 1-D array is valid, selecting zero top values returns empty
arrays, and searching empty sorted data returns position 0. Quickselect still
needs a valid position, and percentile requests must be between 0 and 100.

The invalid calls below are intentional. Catching their expected exceptions
lets **Run All** continue while displaying the messages.

In [ ]:
print("Empty merge sort:", merge_sort([]))
print("Top zero values:", topk(scores, 0)[0])
print("Insertion into empty data:", binary_search([], 10))

invalid_examples = [
    ("Quickselect position outside data", ValueError, lambda: quickselect(scores, len(scores))),
    ("Non-integer top-k count", TypeError, lambda: topk(scores, 2.0)),
    ("Percentile outside [0, 100]", ValueError, lambda: percentile(scores, 101)),
    ("Rank rejects NaN on request", ValueError, lambda: rank([1, np.nan], nan_policy="raise")),
    ("Percentile rejects NaN on request", ValueError, lambda: percentile([1, np.nan], 50, nan_policy="raise")),
    ("Quickselect rejects NaN", ValueError, lambda: quickselect([1, np.nan], 0, rng=17)),
]
for label, expected_error, operation in invalid_examples:
    try:
        operation()
    except expected_error as error:
        print(f"{label}: {type(error).__name__}: {error}")
    else:
        raise AssertionError(f"Expected {expected_error.__name__}: {label}")

print("\nAll demo checks passed.")

## Further exploration

Try changing the scores, the top-k count, the ranking method, or the matrix axis.
Update the expected results if you change an example with an assertion.

The separate benchmark compares implementations against NumPy and Python
references, checking correctness before timing. For a small run, use this command
from the project root:

```sh
python benchmark/benchmark_section2.py --sizes 100 --distributions random duplicates --repeats 3
```

Timing results depend on the machine, input shape, and comparison baseline.
The notebook itself runs only the demonstrations and their checks.